In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)


In [ ]:
### Cell 2 — Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.metrics import mean_squared_error
sns.set(style="darkgrid")

In [ ]:
### Cell 3 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
### Cell 4 — Load Data
def read_file(filename):
    directory = '/content/drive/My Drive/210_capstone/final_datasets/full_hist_attr/'
    return pd.read_parquet(directory + filename + '.parquet')

In [ ]:
df = read_file('full_data_18to23_asofFeb25_1')

# BUILD 2017 BASELINE

In [ ]:
### Cell 4 — Load Data
def read_csv(filename):
    directory = '/content/drive/My Drive/210_capstone/final_datasets/csv/2_12_cleaned_csv/'
    return pd.read_csv(directory + filename + '.csv')

In [ ]:
elec = read_csv('county_electricity_demand')
elec.head(3)

,timestamp,Alameda,Alpine,Amador,Butte,Calaveras,Colusa,Contra Costa,Del Norte,El Dorado,...,Sonoma,Stanislaus,Sutter,Tehama,Trinity,Tulare,Tuolumne,Ventura,Yolo,Yuba
0,2016-01-01 00:00:00+00:00,1176.275794,0.812014,26.598339,162.300902,32.192157,15.268580,1062.423344,25.020711,133.662747,...,358.862354,359.101087,68.594283,46.700827,9.284875,300.275290,41.835854,576.225204,153.936470,53.236849
1,2016-01-01 01:00:00+00:00,1190.669213,0.752825,26.923826,164.286880,32.586073,15.455413,1075.423577,26.634949,135.222954,...,363.253536,378.357263,69.433766,45.640673,9.688812,314.084688,40.801315,605.631208,155.826528,53.893551
2,2016-01-01 02:00:00+00:00,1316.228479,0.832196,29.763021,181.611371,36.022361,17.085228,1188.829870,28.915939,149.481856,...,401.559593,421.932353,76.755706,50.452193,10.849582,357.686718,45.118873,692.614804,172.260811,59.575321


In [ ]:
pop17 = read_csv('pop_housing_data')
pop17.head(3)

,state_county,date_recorded,total_pop,household_pop,group_quarters_pop,total_households,single_detached,single_attached,two_to_four,five_plus,mobile_homes,occupied,vacancy_rate,person_per_household
0,Alameda,1/1/11,1527169,1485580,41589,583581,310375,44511,65370,155509,7816,546259,0.064,2.72
1,Alameda,1/1/12,1549193,1507025,42168,585757,311507,44989,65377,156089,7795,549045,0.063,2.74
2,Alameda,1/1/13,1575139,1531715,43424,589036,312769,45539,65449,157507,7772,554877,0.058,2.76


In [ ]:
import pandas as pd
import numpy as np

# Load your sheets
elec = read_csv('county_electricity_demand')
pop17 = read_csv('pop_housing_data')

# Filter to 2017
elec['timestamp'] = pd.to_datetime(elec['timestamp'], format='mixed')
pop17['date_recorded'] = pd.to_datetime(pop17['date_recorded'], format='mixed')

elec_2017 = elec[elec['timestamp'].dt.year == 2017]
pop_2017 = pop17[pop17['date_recorded'].dt.year == 2017]

# Reshape electricity wide → long
county_cols = [c for c in elec_2017.columns if c != 'timestamp']
elec_long = elec_2017.melt(id_vars=['timestamp'], value_vars=county_cols,
                             var_name='county', value_name='max_daily_electricity')
elec_long['date'] = pd.to_datetime(elec_long['timestamp'].dt.date)

# Reshape population
pop_clean = pop_2017[['state_county', 'date_recorded', 'total_pop']].copy()
pop_clean['county'] = pop_clean['state_county'].str.replace('California', '').str.strip()
pop_clean['date'] = pd.to_datetime(pop_clean['date_recorded'].dt.date)

# Merge
merged = elec_long.merge(pop_clean[['date', 'county', 'total_pop']], on=['date', 'county'], how='left')
merged = merged.sort_values('date')
merged['total_pop'] = merged.groupby('county')['total_pop'].ffill()

# Annual baseline (per capita)
baseline_annual_2017 = merged.groupby('county').apply(
    lambda x: (x['max_daily_electricity'] / x['total_pop']).mean(),
    include_groups=False
).reset_index()
baseline_annual_2017.columns = ['county', 'baseline_mw_per_capita_2017']

# Monthly baseline (pop-weighted)
merged['month'] = merged['date'].dt.month
monthly_baseline = merged.groupby(['county', 'month']).apply(
    lambda x: np.average(x['max_daily_electricity'], weights=x['total_pop']),
    include_groups=False
).reset_index()
monthly_baseline.columns = ['county', 'month', 'baseline_mw_monthly_2017']

print("✓ Annual baseline:")
print(baseline_annual_2017.head())
print("\n✓ Monthly baseline:")
print(monthly_baseline.head(15))

✓ Annual baseline:
      county  baseline_mw_per_capita_2017
0    Alameda                     0.000772
1     Alpine                     0.000745
2     Amador                     0.000783
3      Butte                     0.000792
4  Calaveras                     0.000806

✓ Monthly baseline:
     county  month  baseline_mw_monthly_2017
0   Alameda      1               1229.178341
1   Alameda      2               1208.897972
2   Alameda      3               1143.373823
3   Alameda      4               1142.775683
4   Alameda      5               1255.646994
5   Alameda      6               1429.877871
6   Alameda      7               1521.152816
7   Alameda      8               1502.519448
8   Alameda      9               1400.782646
9   Alameda     10               1176.346007
10  Alameda     11               1147.231107
11  Alameda     12               1192.896290
12   Alpine      1                  0.840117
13   Alpine      2                  0.813094
14   Alpine      3               

In [ ]:
### Cell 5 — Rename Columns
df = df.rename(columns={
    'Date':  'date',
    'County': 'county',
    'BEV':   'bev',
    'PHEV':  'phev',
    'FCEV':  'fcev',
    'Max_Daily_Electricity_Usage': 'max_daily_electricity',
    'Per_Capita_Personal_Income_Adjusted': 'per_cap_income',
    'Total': 'total_ev_charge'
})

## FEATURE ENGINEERING

In [ ]:
### Cell 6 — Target Engineering (BEFORE split)
df['date'] = pd.to_datetime(df['date'])
df['max_elec_per_capita']     = df['max_daily_electricity'] / df['total_pop']
df['max_elec_per_capita_log'] = np.log(df['max_elec_per_capita'])

target = 'max_elec_per_capita_log'

print(df.shape)


(127078, 74)


In [ ]:
ROLLING_SPECS = [
    ("cdd65_pop_roll5",       "cdd65_pop",  5,  "sum"),
    ("hdd65_pop_roll5",       "hdd65_pop",  5,  "sum"),
    ("tmax_k_pop_roll5_max",  "tmax_k_pop", 5,  "max"),
    ("tmax_k_pop_roll7_mean", "tmax_k_pop", 7,  "mean"),

]

def add_rolling_features(df):
    df = df.sort_values(["county", "date"]).copy()
    g = df.groupby("county", sort=False)
    for new_col, src_col, window, agg in ROLLING_SPECS:
        df[new_col] = g[src_col].transform(
            lambda x, w=window, a=agg: getattr(x.rolling(w, min_periods=1), a)()
        )
    return df

In [ ]:
df = add_rolling_features(df)

In [ ]:
df.columns

Index(['date', 'county', 'dpt_afternoon_k_pop', 'hdd65', 'wind_peak_ms_mean',
       'wind_low_ms_mean', 'spfh_peak_kgkg_pop', 'wind_peak_ms_pop', 'cdd75',
       'tavg_k', 'dpt_afternoon_k_mean', 'cloud_cover_pct_pop',
       'dpt_morning_k_mean', 'tmax_k_pop', 'cdd65_pop', 'cloud_cover_pct_mean',
       'tmin_k', 'hdd65_pop', 'dpt_morning_k_pop', 'wind_low_ms_pop',
       'trange_k', 'cdd65', 'tmin_k_pop', 'cdd75_pop', 'spfh_peak_kgkg_mean',
       'tmax_k', 'real_data_urma', 'staying_total', 'entering_total',
       'leaving_total', 'real_data_commuting', 'cuml_count', 'cuml_sq_foot',
       'cuml_utility_cap', 'cuml_dc_load', 'real_data_data_centers',
       'Total_Daily_Electricity_Usage', 'hour_of_max', 'max_daily_electricity',
       'Public Level 1', 'Shared Private Level 1', 'Public Level 2',
       'Shared Private Level 2', 'Public DC Fast', 'Shared Private DC Fast',
       'total_ev_charge', 'real_data_ev_charging', 'bev', 'phev', 'fcev',
       'real_data_ev_poplution', 'pe

In [ ]:
### Dew Point Depression Feature
import numpy as np

def add_dpd_features(df):
    P = 101325  # standard pressure Pa
    w = df['spfh_peak_kgkg_pop']

    # specific humidity → vapor pressure
    e = (w * P) / (0.622 + w)

    # vapor pressure → dew point (K)
    dpt_c = 243.04 * np.log(e / 611.2) / (17.625 - np.log(e / 611.2))
    df['dpt_derived_k'] = dpt_c + 273.15

    # dew point depression (K) — large = dry, small = humid
    df['dpd_k'] = (df['tmax_k_pop'] - df['dpt_derived_k']).clip(lower=0)

    # rolling 5-day mean depression
    df = df.sort_values(['county', 'date'])
    df['dpd_k_roll5'] = (df
        .groupby('county')['dpd_k']
        .transform(lambda x: x.rolling(5, min_periods=1).mean()))

    return df

df = add_dpd_features(df)

#print(f"dpd_k stats:\n{df['dpd_k'].describe()}")

In [ ]:
# Impute per_cap_income — county median, fall back to global median
global_median = df['per_cap_income'].median()

df['per_cap_income'] = (df.groupby('county')['per_cap_income']
    .transform(lambda x: x.fillna(x.median())))

df['per_cap_income'] = df['per_cap_income'].fillna(global_median)

print(df['per_cap_income'].isna().sum())  # should be 0

0


In [ ]:
monthly_baseline.head(2)

,county,month,baseline_mw_monthly_2017
0,Alameda,1,1229.178341
1,Alameda,2,1208.897972


In [ ]:
# === MERGE 2017 BASELINES ===

# Annual baseline
df = df.merge(baseline_annual_2017, on='county', how='left')

# Monthly baseline
df['month'] = df['date'].dt.month
df = df.merge(monthly_baseline, on=['county', 'month'], how='left')

print("✓ Merged 2017 baselines")
print(f"Missing baseline_mw_per_capita_2017: {df['baseline_mw_per_capita_2017'].isna().sum()}")
print(f"Missing baseline_mw_monthly_2017: {df['baseline_mw_monthly_2017'].isna().sum()}")
print(f"\nSample:")
print(df[['county', 'date', 'baseline_mw_per_capita_2017', 'baseline_mw_monthly_2017']].head(10))

✓ Merged 2017 baselines
Missing baseline_mw_per_capita_2017: 0
Missing baseline_mw_monthly_2017: 0

Sample:
    county       date  baseline_mw_per_capita_2017  baseline_mw_monthly_2017
0  Alameda 2018-01-01                     0.000772               1229.178341
1  Alameda 2018-01-02                     0.000772               1229.178341
2  Alameda 2018-01-03                     0.000772               1229.178341
3  Alameda 2018-01-04                     0.000772               1229.178341
4  Alameda 2018-01-05                     0.000772               1229.178341
5  Alameda 2018-01-06                     0.000772               1229.178341
6  Alameda 2018-01-07                     0.000772               1229.178341
7  Alameda 2018-01-08                     0.000772               1229.178341
8  Alameda 2018-01-09                     0.000772               1229.178341
9  Alameda 2018-01-10                     0.000772               1229.178341


In [ ]:
# === QC CHECKS ===

print("="*70)
print("BASELINE QC CHECKS")
print("="*70)

# 1. Check for missing values
print("\n1. Missing Values:")
print(f"   baseline_mw_per_capita_2017: {df['baseline_mw_per_capita_2017'].isna().sum()}")
print(f"   baseline_mw_monthly_2017: {df['baseline_mw_monthly_2017'].isna().sum()}")

# 2. Check that annual baseline is same for all months of same county
print("\n2. Annual baseline consistency (should be identical for all rows of same county):")
consistency = df.groupby('county')['baseline_mw_per_capita_2017'].nunique()
if (consistency == 1).all():
    print("   ✓ PASS - Each county has exactly 1 annual baseline value")
else:
    print("   ✗ FAIL - Some counties have multiple annual baseline values:")
    print(consistency[consistency > 1])

# 3. Check monthly baseline varies by month
print("\n3. Monthly baseline varies by month (Alameda example):")
alameda_monthly = df[df['county'] == 'Alameda'].groupby('month')['baseline_mw_monthly_2017'].mean()
print(alameda_monthly)
print(f"   Min: {alameda_monthly.min():.2f}, Max: {alameda_monthly.max():.2f}")

# 4. Check baseline ranges
print("\n4. Baseline value ranges:")
print(f"   baseline_mw_per_capita_2017: {df['baseline_mw_per_capita_2017'].min():.6f} to {df['baseline_mw_per_capita_2017'].max():.6f}")
print(f"   baseline_mw_monthly_2017: {df['baseline_mw_monthly_2017'].min():.2f} to {df['baseline_mw_monthly_2017'].max():.2f}")

# 5. Check for duplicates (one row per unique county-month-date combo)
print("\n5. Data shape:")
print(f"   Total rows: {len(df)}")
print(f"   Unique counties: {df['county'].nunique()}")
print(f"   Unique dates: {df['date'].nunique()}")
print(f"   Unique county-date combos: {df.groupby(['county', 'date']).ngroups}")

# 6. Sample from different counties
print("\n6. Sample from different counties:")
for county in ['Alameda', 'Alpine', 'Los Angeles', 'Kern']:
    if county in df['county'].values:
        sample = df[df['county'] == county][['county', 'date', 'month', 'baseline_mw_per_capita_2017', 'baseline_mw_monthly_2017']].head(3)
        print(f"\n   {county}:")
        print(sample.to_string(index=False))

BASELINE QC CHECKS

1. Missing Values:
   baseline_mw_per_capita_2017: 0
   baseline_mw_monthly_2017: 0

2. Annual baseline consistency (should be identical for all rows of same county):
   ✓ PASS - Each county has exactly 1 annual baseline value

3. Monthly baseline varies by month (Alameda example):
month
1     1229.178341
2     1208.897972
3     1143.373823
4     1142.775683
5     1255.646994
6     1429.877871
7     1521.152816
8     1502.519448
9     1400.782646
10    1176.346007
11    1147.231107
12    1192.896290
Name: baseline_mw_monthly_2017, dtype: float64
   Min: 1142.78, Max: 1521.15

4. Baseline value ranges:
   baseline_mw_per_capita_2017: 0.000622 to 0.001160
   baseline_mw_monthly_2017: 0.77 to 10057.20

5. Data shape:
   Total rows: 127078
   Unique counties: 58
   Unique dates: 2191
   Unique county-date combos: 127078

6. Sample from different counties:

   Alameda:
 county       date  month  baseline_mw_per_capita_2017  baseline_mw_monthly_2017
Alameda 2018-01-01    

# THE SPLIT

In [ ]:
# Run this before creating datasets — on df before the split
dow_map = {d: i for i, d in enumerate(df['day_of_week'].unique())}
df['day_of_week'] = df['day_of_week'].map(dow_map)

# Re-split
train_df = df[df['date'].dt.year <= 2021].copy()
val_df   = df[df['date'].dt.year == 2022].copy()
test_df  = df[df['date'].dt.year == 2023].copy()

## BUILDING THE TRANSFORMER MODEL

In [ ]:

# ── Feature Groups ─────────────────────────────────────────────────────────────
WEATHER_TIME = [
    'tmax_k_pop', 'tmin_k_pop', 'trange_k',
    'cdd65_pop', 'hdd65_pop', 'cdd75_pop',
    'spfh_peak_kgkg_pop', 'wind_peak_ms_pop', 'dpd_k',
    'month', 'quarter', 'day_of_week', 'is_holiday',
    'baseline_mw_per_capita_2017', 'baseline_mw_monthly_2017'
]  # 15 features → 16 tokens with county

ROLLING_TIME = [
    'cdd65_pop_roll5', 'hdd65_pop_roll5',
    'tmax_k_pop_roll5_max', 'tmax_k_pop_roll7_mean', 'dpd_k_roll5',
    'month', 'quarter', 'day_of_week', 'is_holiday',
    'baseline_mw_per_capita_2017', 'baseline_mw_monthly_2017'
]  # 11 features → 12 tokens with county

GEO_NUMERIC = [
    'total_pop', 'per_cap_income',
    'baseline_mw_per_capita_2017', 'baseline_mw_monthly_2017'
]  # 4 numeric + county embedding

INFRA_TIME = [
    'cuml_count', 'cuml_sq_foot', 'cuml_utility_cap', 'cuml_dc_load', 'bev',
    'month', 'quarter', 'day_of_week', 'is_holiday',
    'baseline_mw_per_capita_2017', 'baseline_mw_monthly_2017'
]  # 11 features → 12 tokens with county

HEAT_WAVE = [
    'baseline_mw_per_capita_2017', 'per_cap_income',
    'cdd75_pop', 'dpd_k_roll5',
    'cdd65_pop'
]

TARGET = 'max_elec_per_capita_log'
COUNTY_EMB_DIM = 8
EMBED_DIM = 32
NUM_HEADS = 8
CROSS_DIM = 64
DROPOUT = 0.1


In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────────
class ClimateDataset(Dataset):
    def __init__(self, df, scalers=None, county_map=None, fit=False):

        w = df[WEATHER_TIME].values.astype(np.float32)
        r = df[ROLLING_TIME].values.astype(np.float32)
        g = df[GEO_NUMERIC].values.astype(np.float32)
        h = df[HEAT_WAVE].values.astype(np.float32)
        i = df[INFRA_TIME].values.astype(np.float32)

        if fit:
            self.scalers = {
                'w': StandardScaler().fit(w),
                'r': StandardScaler().fit(r),
                'g': StandardScaler().fit(g),
                'h': StandardScaler().fit(h),
                'i': StandardScaler().fit(i),
            }
            self.county_map = {c: idx for idx, c in enumerate(df['county'].unique())}
        else:
            self.scalers    = scalers
            self.county_map = county_map

        self.w      = self.scalers['w'].transform(w)
        self.r      = self.scalers['r'].transform(r)
        self.g      = self.scalers['g'].transform(g)
        self.h      = self.scalers['h'].transform(h)
        self.i      = self.scalers['i'].transform(i)
        self.county = df['county'].map(self.county_map).fillna(0).values.astype(np.int64)
        self.y      = df[TARGET].values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.w[idx]),
            torch.tensor(self.r[idx]),
            torch.tensor(self.g[idx]),
            torch.tensor(self.h[idx]),
            torch.tensor(self.i[idx]),
            torch.tensor(self.county[idx]),
            torch.tensor(self.y[idx])
        )

In [ ]:
# ── Attention Block ────────────────────────────────────────────────────────────
class AttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn    = nn.MultiheadAttention(embed_dim, num_heads,
                                              dropout=dropout, batch_first=True)
        self.norm1   = nn.LayerNorm(embed_dim)
        self.ffn     = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
        )
        self.norm2   = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


In [ ]:
# ── Stream Encoder ─────────────────────────────────────────────────────────────
class StreamEncoder(nn.Module):
    """Project features → embed_dim, prepend county token, 2 attention blocks, flatten"""
    def __init__(self, n_features, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, dropout=DROPOUT):
        super().__init__()
        self.proj        = nn.Linear(1, embed_dim)
        self.county_proj = nn.Linear(COUNTY_EMB_DIM, embed_dim)
        self.block1      = AttentionBlock(embed_dim, num_heads, dropout)
        self.block2      = AttentionBlock(embed_dim, num_heads, dropout)
        self.out_dim     = (n_features + 1) * embed_dim  # +1 for county token

    def forward(self, x, county_emb):
        # x: (batch, n_features) → (batch, n_features, embed_dim)
        x = self.proj(x.unsqueeze(-1))
        # county: (batch, 8) → (batch, 1, embed_dim)
        c = self.county_proj(county_emb).unsqueeze(1)
        # prepend county token
        x = torch.cat([c, x], dim=1)   # (batch, n_features+1, embed_dim)
        x = self.block1(x)
        x = self.block2(x)
        return x.flatten(1)


In [ ]:
class ClimateFEAT(nn.Module):
    def __init__(self, n_counties,
                 embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 cross_dim=CROSS_DIM, dropout=DROPOUT):
        super().__init__()

        self.county_emb  = nn.Embedding(n_counties + 1, COUNTY_EMB_DIM)

        # Standard streams: 32 dims
        self.weather_enc = StreamEncoder(len(WEATHER_TIME), embed_dim=32, num_heads=8, dropout=dropout)
        self.rolling_enc = StreamEncoder(len(ROLLING_TIME), embed_dim=32, num_heads=8, dropout=dropout)

        # HEAT WAVE: 64 dims (2x bigger!)
        self.heatwave_enc = StreamEncoder(len(HEAT_WAVE), embed_dim=64, num_heads=8, dropout=dropout)

        self.infra_enc   = StreamEncoder(len(INFRA_TIME), embed_dim=32, num_heads=8, dropout=dropout)

        self.geo_dense   = nn.Sequential(
            nn.Linear(len(GEO_NUMERIC) + COUNTY_EMB_DIM, 32),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Cross-attention (unchanged)
        self.w_to_cross  = nn.Linear(self.weather_enc.out_dim, cross_dim)
        self.r_to_cross  = nn.Linear(self.rolling_enc.out_dim, cross_dim)
        self.g_to_cross  = nn.Linear(32, cross_dim)
        self.h_to_cross  = nn.Linear(self.heatwave_enc.out_dim, cross_dim)  # Now bigger!
        self.cross_attn  = AttentionBlock(cross_dim, num_heads=num_heads, dropout=dropout)

        head_input_dim = 4 * cross_dim + self.infra_enc.out_dim

        self.head = nn.Sequential(
            nn.Linear(head_input_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 64),             nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, w, r, g, h, i, county):
        county_emb = self.county_emb(county)

        w_out = self.weather_enc(w, county_emb)
        r_out = self.rolling_enc(r, county_emb)
        g_out = self.geo_dense(torch.cat([g, county_emb], dim=1))
        h_out = self.heatwave_enc(h, county_emb)  # More capacity!
        i_out = self.infra_enc(i, county_emb)

        w_tok = self.w_to_cross(w_out).unsqueeze(1)
        r_tok = self.r_to_cross(r_out).unsqueeze(1)
        g_tok = self.g_to_cross(g_out).unsqueeze(1)
        h_tok = self.h_to_cross(h_out).unsqueeze(1)

        cross = torch.cat([w_tok, r_tok, g_tok, h_tok], dim=1)
        cross = self.cross_attn(cross)
        cross = cross.flatten(1)

        out = self.head(torch.cat([cross, i_out], dim=1))
        return out.squeeze(-1)

In [ ]:
# ── Datasets & Loaders ─────────────────────────────────────────────────────────
train_ds = ClimateDataset(train_df, fit=True)
val_ds   = ClimateDataset(val_df,
                          scalers=train_ds.scalers,
                          county_map=train_ds.county_map)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=2)



In [ ]:
## Clear Cache
if 'model' in locals():
    del model
torch.cuda.empty_cache()

In [ ]:
# ── Training ───────────────────────────────────────────────────────────────────
device     = torch.device('cuda')
n_counties = len(train_ds.county_map)
model      = ClimateFEAT(n_counties).to(device)
optimizer  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
criterion  = nn.MSELoss()
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {total_params:,}")
print(f"Training on:  {len(train_ds):,} samples")

best_val_loss  = float('inf')
patience_count = 0
PATIENCE       = 30

for epoch in range(150):
    model.train()
    train_losses = []
    for w, r, g, h, i, county, y in train_loader:
        w, r, g, h, i, county, y = [t.to(device) for t in [w, r, g, h, i, county, y]]


        if epoch == 0:  # Only on first epoch
            with torch.no_grad():
                county_emb = model.county_emb(county)
                w_out = model.weather_enc(w, county_emb)
                r_out = model.rolling_enc(r, county_emb)
                g_out = model.geo_dense(torch.cat([g, county_emb], dim=1))
                h_out = model.heatwave_enc(h, county_emb)
                i_out = model.infra_enc(i, county_emb)
                w_tok = model.w_to_cross(w_out).unsqueeze(1)
                r_tok = model.r_to_cross(r_out).unsqueeze(1)
                g_tok = model.g_to_cross(g_out).unsqueeze(1)
                h_tok = model.h_to_cross(h_out).unsqueeze(1)

                cross = torch.cat([w_tok, r_tok, g_tok, h_tok], dim=1)
                print(f"  cross before attn: {cross.shape}")

                cross_out = model.cross_attn(cross)
                print(f"  cross after attn: {cross_out.shape}")

                cross_flat = cross_out.flatten(1)
                print(f"  cross flattened: {cross_flat.shape}")

                final = torch.cat([cross_flat, i_out], dim=1)
                print(f"  final head input: {final.shape}")
                break

        optimizer.zero_grad()
        loss = criterion(model(w, r, g, h, i, county), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for w, r, g, h, i, county, y in val_loader:
            w, r, g, h, i, county, y = [t.to(device) for t in [w, r, g, h, i, county, y]]
            val_losses.append(criterion(model(w, r, g, h, i, county), y).item())

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    scheduler.step(val_loss)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        torch.save(model.state_dict(), 'best_climate_feat.pt')
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(torch.load('best_climate_feat.pt'))
print(f"\nBest val loss: {best_val_loss:.4f}")

Model params: 493,657
Training on:  84,738 samples
  cross before attn: torch.Size([512, 4, 64])
  cross after attn: torch.Size([512, 4, 64])
  cross flattened: torch.Size([512, 256])
  final head input: torch.Size([512, 640])
Epoch   0 | train nan | val 47.0369


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Epoch   5 | train 0.2515 | val 0.0499
Epoch  10 | train 0.2294 | val 0.0216
Epoch  15 | train 0.2225 | val 0.0150
Epoch  20 | train 0.2126 | val 0.0188
Epoch  25 | train 0.2032 | val 0.0461
Epoch  30 | train 0.1991 | val 0.0215
Epoch  35 | train 0.1957 | val 0.0201
Epoch  40 | train 0.1936 | val 0.0266
Epoch  45 | train 0.1931 | val 0.0271
Early stopping at epoch 45

Best val loss: 0.0150


In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for w, r, g, h, i, county, y in val_loader:
        w, r, g, h, i, county = [t.to(device) for t in [w, r, g, h, i, county]]
        preds = model(w, r, g, h, i, county)
        all_preds.append(preds.cpu().numpy())

preds_log = np.concatenate(all_preds)
val_df['preds_mwh_ft2'] = np.exp(preds_log) * val_df['total_pop']
val_df['actual_mwh']    = np.exp(val_df[TARGET]) * val_df['total_pop']

ft2_rmse = np.sqrt(mean_squared_error(val_df['actual_mwh'], val_df['preds_mwh_ft2']))
w       = val_df['total_pop'].values
wrmse   = np.sqrt(np.sum(w * (val_df['actual_mwh'] - val_df['preds_mwh_ft2'])**2) / np.sum(w))
wmean   = np.average(val_df['actual_mwh'], weights=w)
pop_wtd = 100 * wrmse / wmean

print(f"ClimateFEAT Transformer v2:")
print(f"  RMSE MWh:        {ft2_rmse:,.0f}")
print(f"  Pop-wtd RMSE %:  {pop_wtd:.1f}%")

# LA monthly bias
val_df['residual_mwh'] = val_df['actual_mwh'] - val_df['preds_mwh_ft2']
la = val_df[val_df['county'] == 'Los Angeles'].copy()
la['month'] = la['date'].dt.month
print("\nLA Monthly Bias:")
print(la.groupby('month').agg(
    bias_mwh=('residual_mwh', 'mean'),
    rmse_mwh=('residual_mwh', lambda x: np.sqrt((x**2).mean()))
).round(1).to_string())

ClimateFEAT Transformer v2:
  RMSE MWh:        139
  Pop-wtd RMSE %:  12.0%

LA Monthly Bias:
       bias_mwh  rmse_mwh
month                    
1         321.0     743.1
2         115.8     777.7
3         109.1    1006.9
4         157.7     906.6
5         118.8     824.0
6         312.2     576.6
7        -727.4     977.0
8        -367.8     641.8
9        -567.0    1283.3
10        -69.0     652.8
11       -178.4     857.8
12       -732.2    1092.3


In [ ]:
# Check HEAT_WAVE feature correlation with actual demand in Sept
sept_data = val_df[val_df['date'].dt.month == 9].copy()

print("September Heat Wave Features:")
print(f"  cdd75_pop: {sept_data['cdd75_pop'].mean():.1f} (avg) | max: {sept_data['cdd75_pop'].max():.1f}")
print(f"  dpd_k: {sept_data['dpd_k'].mean():.2f} (avg) | min: {sept_data['dpd_k'].min():.2f}")
print(f"  dpd_k_roll5: {sept_data['dpd_k_roll5'].mean():.2f} (avg)")

# Correlation with target
print(f"\nCorrelation with demand (log):")
print(f"  cdd75_pop: {sept_data['cdd75_pop'].corr(sept_data['max_elec_per_capita_log']):.3f}")
print(f"  dpd_k: {sept_data['dpd_k'].corr(sept_data['max_elec_per_capita_log']):.3f}")

# Check LA specifically
la_sept = sept_data[sept_data['county'] == 'Los Angeles']
print(f"\nLA September:")
print(f"  Actual demand range: {la_sept['actual_mwh'].min():.0f} - {la_sept['actual_mwh'].max():.0f}")
print(f"  Prediction range: {la_sept['preds_mwh_ft2'].min():.0f} - {la_sept['preds_mwh_ft2'].max():.0f}")
print(f"  Bias: {(la_sept['preds_mwh_ft2'] - la_sept['actual_mwh']).mean():.0f} MWh")

September Heat Wave Features:
  cdd75_pop: 2.0 (avg) | max: 13.5
  dpd_k: 19.13 (avg) | min: 1.70
  dpd_k_roll5: 19.24 (avg)

Correlation with demand (log):
  cdd75_pop: 0.510
  dpd_k: 0.457

LA September:
  Actual demand range: 9115 - 16651
  Prediction range: 10553 - 15850
  Bias: 567 MWh


In [ ]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 23.7 MB/s eta 0:00:00


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ALL TRAINING ARTIFACTS + LOG TO MLFLOW
# ═══════════════════════════════════════════════════════════════════════════════
#
# Run this ONCE after training while everything is still in memory.
# This saves everything inference needs — no implicit state left behind.
#
# IMPORTANT: day_of_week must still be in its ORIGINAL string form when you
# run this. If you already mapped it to ints, reload the raw data first:
#     df_raw = read_file('full_data_18to23_asofFeb25_1')
#     dow_map = {d: i for i, d in enumerate(df_raw['day_of_week'].unique())}

import mlflow
import joblib
import json
import os
import torch
from datetime import datetime
import pytz

# ── Config ────────────────────────────────────────────────────────────────────
pt = pytz.timezone('America/Los_Angeles')
run_ts = datetime.now(pt).strftime('%m%d_%H%M')
model_version = 'FINAL'
model_name    = f'climatefeat_transformer_{model_version}_{run_ts}'

MODEL_DIR = '/content/drive/My Drive/210_capstone/MLFlow/final_models'
PRED_DIR  = '/content/drive/My Drive/210_capstone/MLFlow/final_predictions'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR,  exist_ok=True)

# ── 1. Reconstruct dow_map from raw data ──────────────────────────────────────
# (safe even if df['day_of_week'] is already int-mapped)
if df['day_of_week'].dtype == object:
    dow_map = {d: i for i, d in enumerate(df['day_of_week'].unique())}
else:
    print("⚠ day_of_week already mapped to int — reloading raw data for dow_map")
    df_raw = read_file('full_data_18to23_asofFeb25_1')
    dow_map = {d: i for i, d in enumerate(df_raw['day_of_week'].unique())}
    del df_raw

print(f"dow_map: {dow_map}")

# ── 2. Feature config (freeze everything inference needs) ─────────────────────
feature_config = {
    'WEATHER_TIME': WEATHER_TIME,
    'ROLLING_TIME': ROLLING_TIME,
    'GEO_NUMERIC':  GEO_NUMERIC,
    'INFRA_TIME':   INFRA_TIME,
    'HEAT_WAVE':    HEAT_WAVE,
    'TARGET':       TARGET,
    'ROLLING_SPECS': [
        ("cdd65_pop_roll5",       "cdd65_pop",  5, "sum"),
        ("hdd65_pop_roll5",       "hdd65_pop",  5, "sum"),
        ("tmax_k_pop_roll5_max",  "tmax_k_pop", 5, "max"),
        ("tmax_k_pop_roll7_mean", "tmax_k_pop", 7, "mean"),
    ],
    'RENAME': {'Date': 'date', 'County': 'county', 'BEV': 'bev'},
    'N_COUNTIES': len(train_ds.county_map),
    'EMBED_DIM':  32,
    'NUM_HEADS':  8,
    'CROSS_DIM':  64,
    'DROPOUT':    0.1,
}

# ── 3. MLflow logging ─────────────────────────────────────────────────────────
mlflow.set_tracking_uri(f'sqlite:///{MODEL_DIR}/mlflow.db')
mlflow.set_experiment('climate-feat-transformer')

with mlflow.start_run(run_name=model_name):

    # Params
    mlflow.log_param('model_version',    model_version)
    mlflow.log_param('embed_dim',        32)
    mlflow.log_param('num_heads',        8)
    mlflow.log_param('dropout',          0.1)
    mlflow.log_param('batch_size',       512)
    mlflow.log_param('lr',               3e-4)
    mlflow.log_param('weight_decay',     1e-3)
    mlflow.log_param('n_weather_feats',  len(WEATHER_TIME))
    mlflow.log_param('n_rolling_feats',  len(ROLLING_TIME))
    mlflow.log_param('n_geo_feats',      len(GEO_NUMERIC))
    mlflow.log_param('n_heatwave_feats', len(HEAT_WAVE))
    mlflow.log_param('n_infra_feats',    len(INFRA_TIME))
    mlflow.log_param('best_epoch',       epoch)

    # Metrics
    mlflow.log_metric('best_val_loss',    best_val_loss)
    mlflow.log_metric('rmse_mwh',         ft2_rmse)
    mlflow.log_metric('pop_wtd_rmse_pct', pop_wtd)

    # ── Save: model weights ───────────────────────────────────────────────
    torch.save(model.state_dict(), f'{MODEL_DIR}/{model_name}.pt')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}.pt')

    # ── Save: scalers + county_map (already existed) ──────────────────────
    joblib.dump(train_ds.scalers,    f'{MODEL_DIR}/{model_name}_scalers.pkl')
    joblib.dump(train_ds.county_map, f'{MODEL_DIR}/{model_name}_county_map.pkl')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}_scalers.pkl')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}_county_map.pkl')

    # ── Save: NEW artifacts for inference ─────────────────────────────────
    joblib.dump(dow_map,        f'{MODEL_DIR}/{model_name}_dow_map.pkl')
    joblib.dump(feature_config, f'{MODEL_DIR}/{model_name}_feature_config.pkl')
    baseline_annual_2017.to_parquet(f'{MODEL_DIR}/baseline_annual_2017.parquet')
    monthly_baseline.to_parquet(f'{MODEL_DIR}/baseline_monthly_2017.parquet')

    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}_dow_map.pkl')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}_feature_config.pkl')
    mlflow.log_artifact(f'{MODEL_DIR}/baseline_annual_2017.parquet')
    mlflow.log_artifact(f'{MODEL_DIR}/baseline_monthly_2017.parquet')

    # ── Save: validation predictions ──────────────────────────────────────
    pred_df = val_df[['county', 'date', 'total_pop', 'actual_mwh',
                       'preds_mwh_ft2', 'residual_mwh']].copy()
    pred_df.to_parquet(f'{PRED_DIR}/{model_name}_preds_val.parquet', index=False)
    mlflow.log_artifact(f'{PRED_DIR}/{model_name}_preds_val.parquet')

    print(f'\n{"="*70}')
    print(f'MLflow run: climate-feat-transformer / {model_name}')
    print(f'{"="*70}')
    print(f'  best_val_loss:    {best_val_loss:.4f}')
    print(f'  rmse_mwh:         {ft2_rmse:,.0f}')
    print(f'  pop_wtd_rmse_pct: {pop_wtd:.1f}%')

# ── 4. Verify all artifacts ──────────────────────────────────────────────────
print(f'\n{"="*70}')
print(f'ARTIFACT VERIFICATION')
print(f'{"="*70}')

artifacts = {
    'model weights':  f'{MODEL_DIR}/{model_name}.pt',
    'scalers':        f'{MODEL_DIR}/{model_name}_scalers.pkl',
    'county_map':     f'{MODEL_DIR}/{model_name}_county_map.pkl',
    'dow_map':        f'{MODEL_DIR}/{model_name}_dow_map.pkl',
    'feature_config': f'{MODEL_DIR}/{model_name}_feature_config.pkl',
    'baseline_annual': f'{MODEL_DIR}/baseline_annual_2017.parquet',
    'baseline_monthly': f'{MODEL_DIR}/baseline_monthly_2017.parquet',
}

all_good = True
for name, path in artifacts.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = '✓' if exists else '✗ MISSING'
    print(f'  {status} {name}: {size:,} bytes')
    if not exists:
        all_good = False

if all_good:
    print(f'\n✅ All 7 artifacts saved. Inference pipeline can run standalone.')
    print(f'   Model name: {model_name}')
else:
    print(f'\n❌ Missing artifacts — fix before running inference.')

⚠ day_of_week already mapped to int — reloading raw data for dow_map
dow_map: {'Sunday': 0, 'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 'Thursday': 4, 'Saturday': 5, 'Friday': 6}

MLflow run: climate-feat-transformer / climatefeat_transformer_FINAL_0315_1948
  best_val_loss:    0.0150
  rmse_mwh:         139
  pop_wtd_rmse_pct: 12.0%

ARTIFACT VERIFICATION
  ✓ model weights: 2,030,149 bytes
  ✓ scalers: 2,585 bytes
  ✓ county_map: 739 bytes
  ✓ dow_map: 101 bytes
  ✓ feature_config: 801 bytes
  ✓ baseline_annual: 3,353 bytes
  ✓ baseline_monthly: 9,921 bytes

✅ All 7 artifacts saved. Inference pipeline can run standalone.
   Model name: climatefeat_transformer_FINAL_0315_1948


In [ ]:
# Where is the model making big mistakes?
val_df['error'] = np.abs(val_df['actual_mwh'] - val_df['preds_mwh_ft2'])
val_df['pct_error'] = 100 * val_df['error'] / val_df['actual_mwh']

print("="*70)
print("WHERE IS THE MODEL STRUGGLING?")
print("="*70)

# 1. By demand level
print("\n1. Error by demand quartile:")
for q in [0.25, 0.5, 0.75, 1.0]:
    threshold = val_df['actual_mwh'].quantile(q)
    subset = val_df[val_df['actual_mwh'] <= threshold]
    rmse = np.sqrt(mean_squared_error(subset['actual_mwh'], subset['preds_mwh_ft2']))
    print(f"   Bottom {q*100:.0f}% (≤ {threshold:,.0f} MWh): RMSE {rmse:,.0f}")

# 2. By county size
print("\n2. Error by county population:")
for pop_threshold in val_df.groupby('county')['total_pop'].mean().quantile([0.25, 0.5, 0.75, 1.0]):
    counties = val_df.groupby('county')['total_pop'].mean()
    big_counties = counties[counties >= pop_threshold].index
    subset = val_df[val_df['county'].isin(big_counties)]
    rmse = np.sqrt(mean_squared_error(subset['actual_mwh'], subset['preds_mwh_ft2']))
    print(f"   Counties ≥ {pop_threshold:,.0f} pop: RMSE {rmse:,.0f}")

# 3. By day of week
print("\n3. Error by day of week:")
dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
for dow in sorted(val_df['day_of_week'].unique()):
    subset = val_df[val_df['day_of_week'] == dow]
    rmse = np.sqrt(mean_squared_error(subset['actual_mwh'], subset['preds_mwh_ft2']))
    bias = (subset['preds_mwh_ft2'] - subset['actual_mwh']).mean()
    print(f"   {dow_names.get(dow, dow)}: RMSE {rmse:,.0f} | Bias {bias:+.0f}")

# 4. By temperature extremes
print("\n4. Error by temperature:")
cold = val_df[val_df['tmin_k_pop'] < val_df['tmin_k_pop'].quantile(0.1)]
normal = val_df[(val_df['tmin_k_pop'] >= val_df['tmin_k_pop'].quantile(0.1)) &
                (val_df['tmax_k_pop'] <= val_df['tmax_k_pop'].quantile(0.9))]
hot = val_df[val_df['tmax_k_pop'] > val_df['tmax_k_pop'].quantile(0.9)]

print(f"   Cold days (tmin < 10th %ile): RMSE {np.sqrt(mean_squared_error(cold['actual_mwh'], cold['preds_mwh_ft2'])):,.0f}")
print(f"   Normal days: RMSE {np.sqrt(mean_squared_error(normal['actual_mwh'], normal['preds_mwh_ft2'])):,.0f}")
print(f"   Hot days (tmax > 90th %ile): RMSE {np.sqrt(mean_squared_error(hot['actual_mwh'], hot['preds_mwh_ft2'])):,.0f}")
# 5. Top 20 worst predictions
print("\n5. Worst 20 predictions:")
worst = val_df.nlargest(20, 'error')[['county', 'date', 'actual_mwh', 'preds_mwh_ft2', 'error', 'tmax_k_pop', 'cdd75_pop']]
print(worst.to_string(index=False))

# 6. By hour of peak (if available)
if 'hour_of_max' in val_df.columns:
    print("\n6. Error by hour of peak demand:")
    for hour in sorted(val_df['hour_of_max'].dropna().unique())[:5]:
        subset = val_df[val_df['hour_of_max'] == hour]
        if len(subset) > 0:
            rmse = np.sqrt(mean_squared_error(subset['actual_mwh'], subset['preds_mwh_ft2']))
            print(f"   Hour {int(hour)}: RMSE {rmse:,.0f} ({len(subset)} days)")

WHERE IS THE MODEL STRUGGLING?

1. Error by demand quartile:
   Bottom 25% (≤ 53 MWh): RMSE 4
   Bottom 50% (≤ 205 MWh): RMSE 11
   Bottom 75% (≤ 664 MWh): RMSE 25
   Bottom 100% (≤ 16,651 MWh): RMSE 139

2. Error by county population:
   Counties ≥ 47,350 pop: RMSE 162
   Counties ≥ 185,183 pop: RMSE 197
   Counties ≥ 697,581 pop: RMSE 271
   Counties ≥ 9,852,298 pop: RMSE 884

3. Error by day of week:
   Mon: RMSE 143 | Bias -2
   Tue: RMSE 118 | Bias -7
   Wed: RMSE 132 | Bias +0
   Thu: RMSE 145 | Bias +6
   Fri: RMSE 151 | Bias -5
   Sat: RMSE 163 | Bias -2
   Sun: RMSE 117 | Bias -5

4. Error by temperature:
   Cold days (tmin < 10th %ile): RMSE 27
   Normal days: RMSE 147
   Hot days (tmax > 90th %ile): RMSE 147

5. Worst 20 predictions:
     county       date   actual_mwh  preds_mwh_ft2       error  tmax_k_pop    cdd75_pop
Los Angeles 2022-09-10  9573.496322   14328.069901 4754.573579  309.560120 4.768180e+00
Los Angeles 2022-10-24  7367.545334    9693.703550 2326.158216  295.6

In [ ]:
# Check if 2022 summer was actually different
summer_2022 = val_df[(val_df['date'].dt.year == 2022) & (val_df['date'].dt.month.isin([7,8,9]))]
print(f"2022 Summer actual mean: {summer_2022['actual_mwh'].mean():.0f}")
print(f"2022 Summer pred mean: {summer_2022['preds_mwh_ft2'].mean():.0f}")

summer_train = train_df[(train_df['date'].dt.month.isin([7,8,9]))]
print(f"Training summer mean: {summer_train['max_daily_electricity'].mean():.0f}")

2022 Summer actual mean: 828
2022 Summer pred mean: 845
Training summer mean: 801


In [ ]:
# Check LA summer vs other counties
la_summer_2022 = summer_2022[summer_2022['county'] == 'Los Angeles']
other_summer_2022 = summer_2022[summer_2022['county'] != 'Los Angeles']

print("LA Summer 2022:")
print(f"  Actual: {la_summer_2022['actual_mwh'].mean():.0f} MWh")
print(f"  Pred: {la_summer_2022['preds_mwh_ft2'].mean():.0f} MWh")
print(f"  Error: {np.sqrt(mean_squared_error(la_summer_2022['actual_mwh'], la_summer_2022['preds_mwh_ft2'])):,.0f} RMSE")

print("\nOther CA Summer 2022:")
print(f"  Actual: {other_summer_2022['actual_mwh'].mean():.0f} MWh")
print(f"  Pred: {other_summer_2022['preds_mwh_ft2'].mean():.0f} MWh")
print(f"  Error: {np.sqrt(mean_squared_error(other_summer_2022['actual_mwh'], other_summer_2022['preds_mwh_ft2'])):,.0f} RMSE")

# Check the outlier heat wave days specifically
la_extreme = la_summer_2022[la_summer_2022['actual_mwh'] > 10000]
print(f"\nLA Extreme days (>10k MWh): {len(la_extreme)} days")
print(f"  Actual: {la_extreme['actual_mwh'].mean():.0f}")
print(f"  Pred: {la_extreme['preds_mwh_ft2'].mean():.0f}")

LA Summer 2022:
  Actual: 12434 MWh
  Pred: 12988 MWh
  Error: 999 RMSE

Other CA Summer 2022:
  Actual: 624 MWh
  Pred: 632 MWh
  Error: 78 RMSE

LA Extreme days (>10k MWh): 85 days
  Actual: 12664
  Pred: 13106


In [ ]:
# Load best model
model.load_state_dict(torch.load('best_climate_feat.pt'))
model.eval()

# Create test dataset
test_ds = ClimateDataset(test_df,
                         scalers=train_ds.scalers,
                         county_map=train_ds.county_map)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2)

# Predictions
all_preds = []
with torch.no_grad():
    for w, r, g, h, i, county, y in test_loader:
        w, r, g, h, i, county = [t.to(device) for t in [w, r, g, h, i, county]]
        preds = model(w, r, g, h, i, county)
        all_preds.append(preds.cpu().numpy())

preds_log = np.concatenate(all_preds)
test_df['preds_mwh_ft2'] = np.exp(preds_log) * test_df['total_pop']
test_df['actual_mwh']    = np.exp(test_df[TARGET]) * test_df['total_pop']

# Metrics
ft2_rmse = np.sqrt(mean_squared_error(test_df['actual_mwh'], test_df['preds_mwh_ft2']))
w       = test_df['total_pop'].values
wrmse   = np.sqrt(np.sum(w * (test_df['actual_mwh'] - test_df['preds_mwh_ft2'])**2) / np.sum(w))
wmean   = np.average(test_df['actual_mwh'], weights=w)
pop_wtd = 100 * wrmse / wmean

print("="*70)
print("ClimateFEAT Transformer — TEST DATA (2023)")
print("="*70)
print(f"  RMSE MWh:        {ft2_rmse:,.0f}")
print(f"  Pop-wtd RMSE %:  {pop_wtd:.1f}%")
print(f"  wmean:  {wmean}")
print(f"  wrmse:  {wrmse}")

# Monthly breakdown
test_df['month'] = test_df['date'].dt.month
test_df['residual_mwh'] = test_df['actual_mwh'] - test_df['preds_mwh_ft2']

print("\n2023 Monthly Performance:")
monthly = test_df.groupby('month').agg({
    'actual_mwh': 'mean',
    'preds_mwh_ft2': 'mean',
    'residual_mwh': ['mean', lambda x: np.sqrt((x**2).mean())]
}).round(0)
monthly.columns = ['actual', 'pred', 'bias', 'rmse']
print(monthly)

# LA specific
print("\n2023 LA Performance:")
la_test = test_df[test_df['county'] == 'Los Angeles']
la_rmse = np.sqrt(mean_squared_error(la_test['actual_mwh'], la_test['preds_mwh_ft2']))
la_bias = (la_test['preds_mwh_ft2'] - la_test['actual_mwh']).mean()
print(f"  Actual mean: {la_test['actual_mwh'].mean():.0f} MWh")
print(f"  Pred mean:   {la_test['preds_mwh_ft2'].mean():.0f} MWh")
print(f"  RMSE: {la_rmse:,.0f} MWh")
print(f"  Bias: {la_bias:+.0f} MWh")

# Compare to validation
print("\n" + "="*70)
print("VAL (2022) vs TEST (2023) Comparison")
print("="*70)
print(f"Validation RMSE:  145 MWh")
print(f"Test RMSE:        {ft2_rmse:,.0f} MWh")
print(f"Validation pop-wtd: 12.4%")
print(f"Test pop-wtd:     {pop_wtd:.1f}%")

print(f"Test pop-wtd:     {pop_wtd:.1f}%")
print(f"Test pop-wtd:     {pop_wtd:.1f}%")

ClimateFEAT Transformer — TEST DATA (2023)
  RMSE MWh:        172
  Pop-wtd RMSE %:  14.8%
  wmean:  3862.5881442033483
  wrmse:  570.0951920111039

2023 Monthly Performance:
       actual   pred  bias   rmse
month                            
1       648.0  623.0  26.0  191.0
2       678.0  638.0  40.0  209.0
3       688.0  636.0  53.0  234.0
4       667.0  629.0  38.0  182.0
5       689.0  653.0  36.0  159.0
6       701.0  682.0  20.0  115.0
7       821.0  850.0 -29.0  147.0
8       826.0  862.0 -36.0  153.0
9       742.0  764.0 -22.0  148.0
10      691.0  678.0  13.0  136.0
11      641.0  631.0   9.0  163.0
12      626.0  629.0  -3.0  192.0

2023 LA Performance:
  Actual mean: 10375 MWh
  Pred mean:   10465 MWh
  RMSE: 1,082 MWh
  Bias: +89 MWh

VAL (2022) vs TEST (2023) Comparison
Validation RMSE:  145 MWh
Test RMSE:        172 MWh
Validation pop-wtd: 12.4%
Test pop-wtd:     14.8%
Test pop-wtd:     14.8%
Test pop-wtd:     14.8%


In [ ]:
# What epoch did it stop at?
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Stopped at epoch: {epoch}")

# Check if val loss was still improving
print(f"\nWas the model still improving when it stopped?")
print(f"Early stopping patience: {PATIENCE}")
print(f"Patience count at stop: {patience_count}")

# The issue might be: your validation set (2022) is quite different from test (2023)
# Let's check the actual data distributions

print("\n2022 (Val) vs 2023 (Test) characteristics:")
print(f"\n2022 Actual demand:")
print(f"  Mean: {val_df['actual_mwh'].mean():.0f}")
print(f"  Std:  {val_df['actual_mwh'].std():.0f}")
print(f"  Summer avg: {val_df[(val_df['date'].dt.month.isin([7,8,9]))]['actual_mwh'].mean():.0f}")
print(f"  Winter avg: {val_df[(val_df['date'].dt.month.isin([12,1,2]))]['actual_mwh'].mean():.0f}")

print(f"\n2023 Actual demand:")
print(f"  Mean: {test_df['actual_mwh'].mean():.0f}")
print(f"  Std:  {test_df['actual_mwh'].std():.0f}")
print(f"  Summer avg: {test_df[(test_df['date'].dt.month.isin([7,8,9]))]['actual_mwh'].mean():.0f}")
print(f"  Winter avg: {test_df[(test_df['date'].dt.month.isin([12,1,2]))]['actual_mwh'].mean():.0f}")

print(f"\n2018-2021 (Train) Summer avg:")
summer_train = train_df[train_df['date'].dt.month.isin([7,8,9])]
print(f"  {summer_train['max_daily_electricity'].mean():.0f} MWh")

Best val loss: 0.0150
Stopped at epoch: 45

Was the model still improving when it stopped?
Early stopping patience: 30
Patience count at stop: 30

2022 (Val) vs 2023 (Test) characteristics:

2022 Actual demand:
  Mean: 703
  Std:  1526
  Summer avg: 828
  Winter avg: 640

2023 Actual demand:
  Mean: 702
  Std:  1519
  Summer avg: 797
  Winter avg: 650

2018-2021 (Train) Summer avg:
  801 MWh
